# Automated Student Grade Processor

This notebook implements 5 automation systems using a real dataset of 2,000 students with scores across 7 subjects, plus metadata like absence days, study hours, and career aspirations.

Automations covered:
  1. Automated email alert system (failing students)
  2. At-risk student detector (absences + performance)
  3. Career path recommender (score-based matching)
  4. Automated report exporter (CSV + summary file)
  5. Interactive grade filter dashboard (sliders + toggles)

In [5]:
!pip install pandas matplotlib seaborn tabulate ipywidgets -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
import datetime, os

# Upload student-scores.csv via Colab sidebar first, then:
df = pd.read_csv("student-scores.csv")

SUBJECTS = ["math_score", "history_score", "physics_score",
            "chemistry_score", "biology_score", "english_score", "geography_score"]

df["average"] = df[SUBJECTS].mean(axis=1).round(2)
df["status"] = df["average"].apply(lambda x: "PASS" if x >= 60 else "FAIL")

print(f"✅ Loaded {len(df)} students | Columns: {list(df.columns)}")


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✅ Loaded 2000 students | Columns: ['id', 'first_name', 'last_name', 'email', 'gender', 'part_time_job', 'absence_days', 'extracurricular_activities', 'weekly_self_study_hours', 'career_aspiration', 'math_score', 'history_score', 'physics_score', 'chemistry_score', 'biology_score', 'english_score', 'geography_score', 'average', 'status']


In [6]:
# --- AUTOMATION 1: Email Alert System ---
# Slider controls the pass/fail threshold. Alerts fire automatically.

def run_email_alerts(threshold):
    df["status"] = df["average"].apply(
        lambda x: "PASS" if x >= threshold else "FAIL"
    )
    failing = df[df["status"] == "FAIL"]
    print(f"Threshold: {threshold}% | Failing: {len(failing)} students\n")
    for _, s in failing.head(5).iterrows():
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] ALERT → {s['email']}")
        print(f"  Subject: Academic Warning — avg {s['average']:.1f}%")
        print(f"  Status : SENT ✅\n")
    if len(failing) > 5:
        print(f"  ... and {len(failing)-5} more alerts sent.")

threshold_slider = widgets.IntSlider(
    value=60, min=40, max=80, step=5,
    description="Pass mark:", continuous_update=False
)
widgets.interact(run_email_alerts, threshold=threshold_slider)

interactive(children=(IntSlider(value=60, continuous_update=False, description='Pass mark:', max=80, min=40, s…

<function __main__.run_email_alerts(threshold)>

In [7]:
# --- AUTOMATION 2: At-Risk Detector ---
# Flags students based on multiple risk factors. Toggles enable/disable each factor.

def detect_at_risk(use_absence, use_score, use_study, absence_thresh, score_thresh, hours_thresh):
    risk = pd.Series([False] * len(df))
    if use_absence:
        risk |= df["absence_days"] > absence_thresh
    if use_score:
        risk |= df["average"] < score_thresh
    if use_study:
        risk |= df["weekly_self_study_hours"] < hours_thresh
    at_risk = df[risk]
    print(f"⚠️  {len(at_risk)} students flagged as at-risk ({len(at_risk)/len(df)*100:.1f}%)")
    display(at_risk[["first_name","last_name","absence_days","average","weekly_self_study_hours"]].head(8))

widgets.interact(detect_at_risk,
    use_absence=widgets.Checkbox(value=True, description="High absences"),
    use_score=widgets.Checkbox(value=True, description="Low average score"),
    use_study=widgets.Checkbox(value=False, description="Low study hours"),
    absence_thresh=widgets.IntSlider(value=7, min=1, max=20, description="Max absences:"),
    score_thresh=widgets.IntSlider(value=55, min=30, max=75, description="Min score:"),
    hours_thresh=widgets.IntSlider(value=5, min=0, max=51, description="Min weekly study hours:")
)


interactive(children=(Checkbox(value=True, description='High absences'), Checkbox(value=True, description='Low…

<function __main__.detect_at_risk(use_absence, use_score, use_study, absence_thresh, score_thresh, hours_thresh)>

In [8]:
# --- AUTOMATION 3: Career Path Recommender ---
# Maps each student's top subjects to a recommended career automatically.

CAREER_MAP = {
    "math_score":       ["Software Engineer", "Accountant", "Stock Investor"],
    "physics_score":    ["Scientist", "Construction Engineer"],
    "chemistry_score":  ["Doctor", "Scientist"],
    "biology_score":    ["Doctor", "Scientist"],
    "history_score":    ["Lawyer", "Government Officer", "Teacher"],
    "english_score":    ["Writer", "Teacher", "Lawyer"],
    "geography_score":  ["Real Estate Developer", "Government Officer"],
}

def recommend_career(row):
    top_subj = row[SUBJECTS].idxmax()
    return CAREER_MAP[top_subj][0]

df["recommended_career"] = df.apply(recommend_career, axis=1)
match = (df["career_aspiration"] == df["recommended_career"]).mean() * 100

print(f"Career match accuracy: {match:.1f}%")
display(df[["first_name","career_aspiration","recommended_career"]].head(10))

career_filter = widgets.Dropdown(
    options=["All"] + list(df["career_aspiration"].unique()),
    description="Filter career:"
)
def show_career(career):
    subset = df if career == "All" else df[df["career_aspiration"]==career]
    display(subset[["first_name","career_aspiration","recommended_career","average"]].head(8))
widgets.interact(show_career, career=career_filter)

Career match accuracy: 9.7%


,first_name,career_aspiration,recommended_career
0,Paul,Lawyer,Doctor
1,Danielle,Doctor,Doctor
2,Tina,Government Officer,Lawyer
3,Tara,Artist,Doctor
4,Anthony,Unknown,Software Engineer
5,Kelly,Unknown,Lawyer
6,Anthony,Software Engineer,Software Engineer
7,George,Software Engineer,Software Engineer
8,Stanley,Unknown,Software Engineer
9,Audrey,Teacher,Software Engineer


interactive(children=(Dropdown(description='Filter career:', options=('All', 'Lawyer', 'Doctor', 'Government O…

<function __main__.show_career(career)>

In [9]:
# --- AUTOMATION 4: Automated Report Exporter ---
# Generates a formatted summary .txt report + filtered CSVs automatically.

def export_report(include_failing, include_atrisk, include_careers):
    lines = []
    lines.append("="*50)
    lines.append(f"STUDENT GRADE REPORT — {datetime.date.today()}")
    lines.append("="*50)
    lines.append(f"Total students  : {len(df)}")
    lines.append(f"Overall average : {df['average'].mean():.2f}%")
    lines.append(f"Pass rate       : {(df.status=='PASS').mean()*100:.1f}%")

    if include_failing:
        failing = df[df["status"]=="FAIL"]
        failing.to_csv("failing_students.csv", index=False)
        lines.append(f"\n[FAILING] {len(failing)} students → saved to failing_students.csv")

    if include_careers:
        summary = df.groupby("recommended_career")["average"].mean().round(1)
        lines.append("\n[CAREERS] Avg score by recommended career:")
        for career, avg in summary.items():
            lines.append(f"  {career:<20} {avg}%")

    report = "\n".join(lines)
    with open("grade_report.txt", "w") as f:
        f.write(report)
    print(report)
    print("\n✅ grade_report.txt saved to Colab files.")

widgets.interact(export_report,
    include_failing=widgets.Checkbox(value=True, description="Export failing list"),
    include_atrisk=widgets.Checkbox(value=True, description="Include at-risk summary"),
    include_careers=widgets.Checkbox(value=False, description="Include career breakdown")
)

interactive(children=(Checkbox(value=True, description='Export failing list'), Checkbox(value=True, descriptio…

<function __main__.export_report(include_failing, include_atrisk, include_careers)>

In [10]:
# --- AUTOMATION 5: Interactive Visual Dashboard ---
# Filters update all 4 charts simultaneously.

def dashboard(gender, career, min_avg, show_parttime):
    subset = df.copy()
    if gender != "All":
        subset = subset[subset["gender"] == gender]
    if career != "All":
        subset = subset[subset["career_aspiration"] == career]
    subset = subset[subset["average"] >= min_avg]
    if show_parttime:
        subset = subset[subset["part_time_job"] == True]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(f"Dashboard — {len(subset)} students", fontsize=14)

    # Panel 1: Score distribution
    axes[0,0].hist(subset["average"], bins=20, color="steelblue")
    axes[0,0].set_title("Score Distribution")

    # Panel 2: Subject averages
    subj_labels = [s.replace("_score","").capitalize() for s in SUBJECTS]
    axes[0,1].bar(subj_labels, subset[SUBJECTS].mean(), color="coral")
    axes[0,1].set_title("Subject Averages"); axes[0,1].tick_params(axis="x", rotation=30)

    # Panel 3: Top careers pie
    top5 = subset["career_aspiration"].value_counts().head(5)
    axes[1,0].pie(top5, labels=top5.index, autopct="%1.0f%%")
    axes[1,0].set_title("Top Career Aspirations")

    # Panel 4: Study hours vs average scatter
    axes[1,1].scatter(subset["weekly_self_study_hours"], subset["average"], alpha=0.3)
    axes[1,1].set_title("Study Hours vs Average Score")
    axes[1,1].set_xlabel("Study hrs/week"); axes[1,1].set_ylabel("Average %")

    plt.tight_layout(); plt.show()

widgets.interact(dashboard,
    gender=widgets.Dropdown(options=["All","male","female"], description="Gender:"),
    career=widgets.Dropdown(options=["All"]+list(df.career_aspiration.unique()), description="Career:"),
    min_avg=widgets.IntSlider(value=0, min=50, max=100, step=1, description="Min avg:"),
    show_parttime=widgets.Checkbox(value=False, description="Part-time only")
)

interactive(children=(Dropdown(description='Gender:', options=('All', 'male', 'female'), value='All'), Dropdow…

<function __main__.dashboard(gender, career, min_avg, show_parttime)>

## Reflection

This project taught us that automation is not a single task — it is a pipeline. By building five interconnected automation systems on a real student dataset, we learned how each layer adds value: the email alert system replaces manual grade-checking; the at-risk detector surfaces patterns invisible to a teacher reviewing one row at a time; the career recommender shows how rule-based AI can be applied to real data; the report exporter demonstrates how automation produces persistent, shareable outputs; and the interactive dashboard proves that automation does not have to be invisible — it can be a live tool that educators interact with.

We also learned the importance of parameterization. Rather than hardcoding thresholds, we used sliders and toggles so that the system adapts to different grading policies. This reflects how real-world automation systems are built — configurable, not rigid. Working with a real 2,000-student dataset instead of simulated data added credibility and introduced us to real-world data cleaning considerations. Overall, this assignment reinforced that the goal of automation is to remove repetitive work and surface actionable insights.